# Exploration 02 — data cleaning and feature engineering

> **Exploratory** (CLAUDE.md, "Exploration"). **Data:** the training pool P only. The cleaning rules are
> **unsupervised**: they look at the features, never at |μ|, so applying them across the pool cannot leak label
> information into the small-N learning curves of notebook 04.

**Part 2 of the classical analysis.** Input: the EDA's findings (`results/explore01_recommendations.csv`).
Output, used by every later notebook:
- `data/processed/explore_features.parquet`: one row per pool molecule, every feature, the target and bookkeeping
  (regenerated by this notebook, not committed);
- `results/explore02_feature_catalog.csv`: one row per feature, with its block, whether it is headline-legal,
  whether it was kept, and why it was dropped if not.

Thresholds come from `configs/explore.yaml`; change them there and re-run 02 → 05.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from qm9dipole import REPO_ROOT
from qm9dipole.data import load_qm9_table
from qm9dipole.explore import CATALOG_NAME, DFT_BLOCKS, FEATURES_PATH, LEGAL_BLOCKS, feature_sets, load_config
from qm9dipole.plots import AXIS, INK_SECONDARY, MUTED, ROLE_COLORS, ROLE_LABELS, SEQUENTIAL, SERIES, SURFACE, use_style
from qm9dipole.provenance import RESULTS_DIR, save_result
from qm9dipole.splits import load_splits, training_pool

use_style()
FIG_DIR = REPO_ROOT / "figures"
CFG = load_config()

table = load_qm9_table()
splits = load_splits()
P = table.set_index("id").loc[training_pool(splits, table)].reset_index()
assert not set(P["id"]) & (set(splits.test_familiar) | set(splits.test_unseen))
mol = P.set_index("id")
y = mol["mu"]
train_ids = np.unique(np.concatenate([ids for by_n in splits.train.values() for ids in by_n.values()]))
print(f"training pool P: {len(P):,} molecules; training sets S_(s,N) use {len(train_ids):,} of them")

recommendations = pd.read_csv(RESULTS_DIR / "explore01_recommendations.csv")
print("\nWhat the EDA (notebook 01) asks this notebook to do:")
display(recommendations[recommendations["notebook"] == 2].drop(columns="notebook").style.hide(axis="index"))

## 1. Record-level cleaning

### 1a. Doubled frequency lists
25 raw records print their frequency list twice, with identical halves. Keep the first copy.

In [ ]:
from qm9dipole.cleaning import clean_frequencies, is_linear, vibrational_modes

freqs = pd.Series([clean_frequencies(f, n) for f, n in zip(mol["freqs"], mol["n_atoms"])], index=mol.index)
changed = mol["freqs"].map(len) != freqs.map(len)
linear = pd.Series([is_linear(f, n) for f, n in zip(freqs, mol["n_atoms"])], index=mol.index)
assert all(len(f) == vibrational_modes(n, lin) for f, n, lin in zip(freqs, mol["n_atoms"], linear))
print(f"de-doubled: {changed.sum()} lists; every molecule now lists exactly 3n − 6 modes (3n − 5 for the "
      f"{linear.sum()} linear ones: {', '.join(mol.loc[linear, 'smiles'])})")

### 1b. Rotational constants → moments of inertia
The EDA showed A, B, C = 505.379 GHz·amu·Å² / I, with I the principal moments of inertia computed from Z and R, and
QM9 storing A = 0 for the infinite constant of a linear molecule. The moments carry the same information, are finite
for every molecule, and are far less skewed. They enter as engineered features computed from Z and R
(`features.inertia_moments`), so they stay legal for headline models. QM9's A, B, C columns are not used.

In [ ]:
from qm9dipole.cleaning import moments_from_rotational_constants
from qm9dipole.features import inertia_moments

from_qm9 = np.sort(moments_from_rotational_constants(mol[["A", "B", "C"]].to_numpy()), axis=1)
from_geometry = np.array([inertia_moments(z, r) for z, r in zip(mol["Z"], mol["R"])])
rel = np.abs(from_geometry - from_qm9).max(axis=1) / from_geometry.max(axis=1)
skews = pd.DataFrame({"rotational constant (GHz)": mol[["A", "B", "C"]].skew().to_numpy(),
                      "moment of inertia (amu·Å²)": pd.DataFrame(from_geometry).skew().to_numpy()},
                     index=["A ↔ smallest moment", "B ↔ middle", "C ↔ largest"])
print(f"moments from Z, R vs from QM9's A, B, C: max relative difference {rel.max():.1e} "
      f"(linear molecules: smallest moment {from_geometry[linear.to_numpy(), 0].max():.1e} amu·Å²)")
display(skews.style.format("{:.1f}").set_caption("skewness: the reciprocal tames the tail"))

### 1c. Duplicate molecules
12 pairs of pool molecules are the same molecule (identical geometry and |μ|). No training set holds both twins,
and `tests/test_splits.py` guards that, so the fixed training sets need no change. Every molecule keeps its row in
the feature table, so all training sets stay complete. A flag `pool_analysis` keeps one twin per pair, preferring
one that is in a training set, for any analysis that runs over the whole pool.

In [ ]:
from qm9dipole.cleaning import duplicate_groups
from qm9dipole.descriptors import cm_spectrum

spectra = np.stack([cm_spectrum(z, r) for z, r in zip(mol["Z"], mol["R"])])
groups = duplicate_groups(mol.index.to_numpy(), mol["formula"], spectra, CFG["cleaning"]["duplicate_cm_decimals"])
pool_analysis = pd.Series(True, index=mol.index, name="pool_analysis")
for g, members in groups.groupby(groups):
    ids = members.index.to_numpy()
    in_train = [i for i in ids if i in set(train_ids)]
    keep = in_train[0] if in_train else ids.min()
    pool_analysis.loc[[i for i in ids if i != keep]] = False
print(f"{len(groups)} molecules in {groups.nunique()} duplicate groups -> {(~pool_analysis).sum()} flagged "
      f"pool_analysis = False; {pool_analysis.sum():,} molecules for pool-wide analyses")

## 2. Feature engineering

Six blocks. The first three are **functions of Z and R** (headline-legal); the last three read **DFT outputs**
(exploration-only, and every result that uses them is labeled):

| Block | Features | Source |
|---|---|---|
| `composition` | counts of C, H, N, O, F | Z |
| `cm` | 29 Coulomb-matrix eigenvalues (M2) | Z, R |
| `engineered` | geometry, bond-dipole proxies, bond counts, **bond orders** (EDA §7b), **ring count**, **moments of inertia** (§1b) | Z, R + tabulated radii, electronegativities, isotope masses |
| `dft_property` | α, HOMO, LUMO, gap, ⟨R²⟩, ZPVE, U0, U, H, G, Cv | DFT |
| `charge` | Mulliken point-charge dipole (D), max \|q\|, charge separation | DFT charges |
| `frequency` | lowest/highest/mean mode, X–H stretches, soft modes | DFT frequencies (cleaned, §1a) |

In [ ]:
from qm9dipole.descriptors import feature_frame
from qm9dipole.eda import DATA_DICTIONARY
from qm9dipole.features import (CHARGE_FEATURE_NAMES, FREQUENCY_FEATURE_NAMES, charge_features,
                                frequency_features)

DFT_PROPERTIES = ["alpha", "homo", "lumo", "gap", "r2", "zpve", "U0", "U", "H", "G", "Cv"]
blocks = {
    "composition": feature_frame(P, "composition"),
    "cm": feature_frame(P, "cm_spectrum"),
    "engineered": feature_frame(P, "engineered"),
    "dft_property": mol[DFT_PROPERTIES].copy(),
    "charge": pd.DataFrame([charge_features(q, r) for q, r in zip(mol["q"], mol["R"])],
                           index=mol.index, columns=list(CHARGE_FEATURE_NAMES)),
    "frequency": pd.DataFrame([frequency_features(f) for f in freqs], index=mol.index,
                              columns=list(FREQUENCY_FEATURE_NAMES)),
}
for name, frame in blocks.items():
    frame.index.name = "id"
    print(f"{name:13s} {frame.shape[1]:3d} features  ({'legal' if name in LEGAL_BLOCKS else 'DFT, exploration only'})")
all_features = pd.concat(blocks.values(), axis=1)
assert all_features.columns.is_unique and all_features.index.equals(mol.index)

### 2a. Are the engineered features invariant?
Rotation, translation and atom relabeling must leave every legal feature unchanged (PLAN §8).

In [ ]:
from qm9dipole.invariance import invariance_table

sample = P.sample(100, random_state=2026)
inv = invariance_table(list(zip(sample["Z"], sample["R"])), seed=2026, descriptors=("engineered",))
assert inv["invariant"].all()
display(inv.style.format({"mean_change": "{:.1e}", "max_change": "{:.1e}"}).hide(axis="index"))

### 2b. Do the engineered features add information?
Mutual information with |μ| for every legal non-spectrum feature, next to the strongest given fields from the EDA.
The question: does turning geometry into chemistry (polar bonds, their directions, bond orders) recover some of
what the DFT charges know?

In [ ]:
from sklearn.feature_selection import mutual_info_regression

legal_named = pd.concat([blocks["composition"], blocks["engineered"]], axis=1)
legal_named = legal_named.loc[:, legal_named.nunique() > 1]
mi = pd.Series(mutual_info_regression(legal_named.to_numpy(), y.to_numpy(), random_state=0), index=legal_named.columns)
rho = legal_named.corrwith(y, method="spearman")
top = mi.sort_values(ascending=False).head(18)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"width_ratios": [1, 1.25]})
ax0.barh(top.index[::-1], top.values[::-1], color=ROLE_COLORS["legal"], height=0.7)
ax0.set(xlabel="mutual information with |μ| (nats)", title="Legal features: the 18 most informative")
ax0.grid(axis="y", visible=False)
dipoles = pd.DataFrame({"bond_dipole_norm (legal)": blocks["engineered"]["bond_dipole_norm"],
                        "en_point_dipole (legal)": blocks["engineered"]["en_point_dipole"],
                        "charge_dipole_D (DFT)": blocks["charge"]["charge_dipole_D"]})
dmi = pd.Series(mutual_info_regression(dipoles.to_numpy(), y.to_numpy(), random_state=0), index=dipoles.columns)
ax1.bar(range(3), dmi.values, color=[ROLE_COLORS["legal"], ROLE_COLORS["legal"], ROLE_COLORS["dft"]], width=0.6)
ax1.set_xticks(range(3), [c.replace(" (", "\n(") for c in dmi.index])
for k, v in enumerate(dmi.values):
    ax1.annotate(f"{v:.2f}", (k, v), xytext=(0, 3), textcoords="offset points", ha="center", color=INK_SECONDARY)
ax1.set(ylabel="mutual information with |μ| (nats)", title="Three dipole estimates: geometry only vs DFT charges")
ax1.grid(axis="x", visible=False)
fig.savefig(FIG_DIR / "explore02_engineered_information.png")
plt.show()
print(f"best given legal field in the EDA: n_H, MI ≈ 0.18 nats; best engineered: {mi.idxmax()}, MI {mi.max():.2f}")

In [ ]:
orders = blocks["engineered"][["triple_C-N", "double_C-O", "triple_C-C"]] > 0
labels = {"triple_C-N": "nitrile C≡N", "double_C-O": "carbonyl C=O", "triple_C-C": "alkyne C≡C"}
fig, ax = plt.subplots(figsize=(8.5, 3.4))
data, ticks = [], []
for col, name in labels.items():
    for has in (False, True):
        data.append(y[orders[col] == has])
        ticks.append(f"{'with' if has else 'without'}\n{name}\n(n={int((orders[col] == has).sum()):,})")
style = dict(color=MUTED, linewidth=1)
ax.boxplot(data, tick_labels=ticks, widths=0.55, showfliers=False, boxprops=style, whiskerprops=style, capprops=style,
           medianprops=dict(color=ROLE_COLORS["legal"], linewidth=1.75))
ax.set(ylabel="|μ| (D)", title="|μ| with and without each bond type read from bond lengths")
ax.tick_params(axis="x", labelsize=8)
fig.savefig(FIG_DIR / "explore02_bond_orders.png")
plt.show()

## 3. Feature-level cleaning (unsupervised)

Two rules from `configs/explore.yaml`, applied **within** groups so that a legal feature is never dropped because a
DFT feature duplicates it:
1. **zero variance:** a column that never changes in the pool carries no information;
2. **redundancy:** of two columns with |Spearman ρ| ≥ 0.99 (the same information up to a monotone transform), keep
   the one earlier in priority order: composition before engineered; within the DFT group, block order.

The Coulomb spectrum (`cm`) is exempt from rule 2. It is a fixed descriptor (PLAN §5), and its 8-qubit compression
by PCA (notebook 03) already merges correlated eigenvalues.

Exact **linear** dependencies, such as n_H = Σ (H-bond counts) or ring count = bonds − atoms + 1, are kept on
purpose. Ridge regularization handles collinearity for linear models, and trees cannot form differences, so a direct
count such as O–H bonds is worth keeping for them. Notebook 03 measures the collinearity.

In [ ]:
from qm9dipole.cleaning import redundant_columns, zero_variance

catalog_rows = []
dropped = {}
threshold = CFG["cleaning"]["redundancy_spearman"]
groups_for_rules = {"legal (composition + engineered)": ["composition", "engineered"], "cm": ["cm"],
                    "DFT": list(DFT_BLOCKS)}
for group, members in groups_for_rules.items():
    frame = pd.concat([blocks[b] for b in members], axis=1)
    pool_frame = frame.loc[pool_analysis]
    for col in zero_variance(pool_frame):
        dropped[col] = "zero variance in the pool"
    if group != "cm":
        live = [c for c in frame.columns if c not in dropped]
        for r in redundant_columns(pool_frame[live], threshold).itertuples():
            dropped[r.dropped] = f"redundant with {r.kept} (|ρ| = {r.rho:.4f})"

meaning = {row[0]: row[3] for row in DATA_DICTIONARY}
for b, frame in blocks.items():
    for col in frame.columns:
        catalog_rows.append({"feature": col, "block": b, "legal": b in LEGAL_BLOCKS, "kept": col not in dropped,
                             "reason": dropped.get(col, ""), "description": meaning.get(col, "")})
catalog = pd.DataFrame(catalog_rows)
summary = catalog.groupby("block", sort=False).agg(features=("feature", "size"), kept=("kept", "sum"))
display(summary.T)
display(catalog[~catalog["kept"]][["feature", "block", "reason"]].style.hide(axis="index"))

## 4. Sanity checks

In [ ]:
features = all_features[catalog.loc[catalog["kept"], "feature"]].copy()
assert np.isfinite(features.to_numpy()).all(), "NaN or inf in the feature table"
assert set(train_ids) <= set(features.index), "a training-set molecule is missing"
assert features.index.is_unique
assert not catalog.loc[catalog["block"].isin(DFT_BLOCKS), "legal"].any(), "a DFT feature is marked legal"
sets = feature_sets(catalog)
for name, cols in sets.items():
    legal = catalog.set_index("feature").loc[cols, "legal"].all()
    print(f"{name:15s} {len(cols):3d} features  {'headline-legal' if legal else 'EXPLORATION ONLY (uses DFT outputs)'}")
assert all(catalog.set_index("feature").loc[sets[s], "legal"].all() for s in ("composition", "cm", "structure", "cm+structure"))
print("no NaN/inf; every training-set molecule present; DFT features never in a legal set")

## 5. Save for the next notebooks

In [ ]:
out = features.join(mol[["mu", "formula", "n_heavy"]]).join(pool_analysis)
FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(FEATURES_PATH)
save_result(catalog, CATALOG_NAME, config=CFG["cleaning"], n_molecules=len(out),
            label="exploratory; feature catalog for the training pool P")
print(f"wrote {FEATURES_PATH.relative_to(REPO_ROOT).as_posix()}: {out.shape[0]:,} molecules × {features.shape[1]} features")
print(f"wrote results/{CATALOG_NAME}.csv: {len(catalog)} features, {catalog['kept'].sum()} kept")

## Summary

1. **Record-level cleaning.** 25 doubled frequency lists were cut to one copy. The rotational constants were
   replaced by moments of inertia computed from Z and R, which agree with QM9 to 1.2 × 10⁻⁵ and bring skewness down
   from 73 to 0.4. 12 duplicate twins are flagged for pool-wide analyses; the training sets are untouched.
2. **Feature engineering works where chemistry says it should.** The bond-dipole vector sum is the most informative
   legal feature (MI 0.37 nats, twice the best given field, n_H at 0.18), and the four polarity features lead the
   ranking. Bond orders read from bond lengths behave as expected: nitriles raise the median |μ| from 2.4 to 4.1 D
   and carbonyls from 2.2 to 3.3 D, while non-polar alkynes change nothing. All 33 engineered features are
   invariant to 4.5 × 10⁻¹³.
3. **The DFT charges still know three times more** (MI 1.11 nats). That is the gap a structure-only model has to
   close by combining features.
4. **Cleaning dropped 10 of 86 features, each with a recorded reason:** `bonds_C-F` (identical to n_F: every F in
   QM9 sits on a carbon), six bond types that never occur, and U, H, G (identical to U0 in rank).
5. **Feature sets for notebooks 03 → 05:** composition (5), cm (29), structure (31) and cm+structure (60), all
   headline-legal; structure+dft (47), exploration-only.

**Carried forward:** notebook 03 decides how to scale these features (from the EDA's tails and these count
features), and measures how many independent directions they really span.